#Agent harness и навыки агента

> Внимание! Материал ноутбука подходит для работы в Google Colaboratory. Мы не можем гарантировать стабильную работу кода на личных устройствах и на других системах виртуализации.

##Подготовка окружения

Установим необходимые библиотеки

In [1]:
# %pip install -q "deepagents>=0.5,<0.6" "gigachat>=0.2,<0.3" "langchain>=1.3,<2" "langchain-gigachat>=0.5,<0.6" "langgraph>=1.2,<2" "python-dotenv>=1,<2"

#  Добавьте GIGACHAT_CREDENTIALS в панель Colab «Секреты» и разрешите
#  notebook доступ к нему. Затем раскомментируйте следующие три строки.

import os
# from google.colab import userdata
# os.environ["GIGACHAT_CREDENTIALS"] = userdata.get("GIGACHAT_CREDENTIALS")
from dotenv import load_dotenv
load_dotenv()

from pprint import pprint

### [Скачайте](https://drive.google.com/file/d/1CcFJOvorKYgNtGagSHjUrlwQKLivUVGI/view?usp=sharing) и сохраните папку в корень проекта

In [ ]:
! unzip -q skills.zip

## Введение

### Постановка проблемы

В первых двух занятиях модель выбирала Python-инструменты. Теперь добавим два понятия:

- **навык (`skill`)** — инструкция, когда и как решать задачу определённого типа;
- **дочерний агент (`subagent`)** — отдельный агент для узкой части задачи.

### Зачем они нужны

Если поместить все правила и инструменты в одного агента, его инструкция быстро разрастается, выбор действий становится менее точным, а каждый запрос получает лишний контекст и полномочия.

- **Навык** хранит предметные инструкции и справочные материалы. Он помогает подключать только нужные знания, например правила возврата, не запуская отдельный цикл модели.
- **Дочерний агент** выполняет самостоятельную часть работы со своим контекстом, набором инструментов и бюджетом. Например, `billing-researcher` проверяет правило возврата, не получая все инструменты управляющего агента.

Коротко: навык отвечает на вопрос «какие знания и правила применить», а дочерний агент — «кому поручить отдельную часть задачи». Навык можно использовать без дочернего агента; дочернему агенту, напротив, часто назначают один узкий навык.

Во втором занятии один агент уже умел выбирать следующий инструмент по наблюдению. Здесь сохраним этот принцип, но разделим большую задачу: нужные знания оформим как навык, а узкое самостоятельное исследование поручим дочернему агенту.

**Agent harness (контур выполнения агента)** — это код приложения вокруг модели: он принимает предложенные действия, проверяет их и только затем запускает инструменты или дочерних агентов. Такой контур нужен, потому что системная инструкция сама по себе не обеспечивает разрешения, бюджеты, повторы и регистрацию событий.

В этом занятии мы соберём небольшой учебный `AgentHarness`: он выбирает версию навыка, проверяет запрос к инструменту, считает шаги, ограниченно повторяет временный сбой, отделяет внешний эффект, запускает дочернего агента и записывает безопасный `trace`. Затем сопоставим эти границы с графом Deep Agents на GigaChat. Учебные классы ниже не являются внутренними классами Deep Agents: они делают правила явными до знакомства с библиотечным API.

### Цели занятия

Научиться описывать версионируемый навык с условиями срабатывания и ресурсами, разрешать их конфликты, расширять контур реестрами и подключаемыми обработчиками (`hooks`), применять бюджеты шагов и повторов, ограничивать рабочую область и отделять предложение операции от внешнего побочного эффекта.

### Предварительные знания

Достаточно понимать Python-функции, `dataclass` и Function Calling из первых двух занятий.

### Итоговый артефакт

Вы получите исполняемый `AgentHarness` с реестрами навыков, инструментов и дочерних агентов, бюджетом шагов, повторами, подключаемыми обработчиками, `tracing`, рабочей областью и предложениями операций. Затем увидите библиотечную реализацию тех же границ в Deep Agents. Сетевой граф содержит только прикладные инструменты для чтения.

## Карта занятия

1. Создать карточку навыка billing.
2. Выбрать навык по словам в запросе.
3. Проверить запрос к инструменту по списку разрешений и схеме JSON.
4. Подготовить поручение (`Handoff`) и проверить результат дочернего агента.
5. Прочитать готовый локальный проход, который соединяет первые четыре шага.
6. Разобрать готовую границу файловой рабочей области.
7. Связать бюджеты, повторы, подключаемые обработчики, побочные эффекты и запуск дочернего агента в `AgentHarness`.
8. Реализовать те же границы через граф Deep Agents с GigaChat.

### Карта зависимостей

Не нужно удерживать все классы одновременно. Каждая строка добавляет одну границу, а стрелка показывает, кто использует результат предыдущего шага:

```text
SkillRegistry
    ↓ выбранный Skill
ToolRegistry → validate_tool_request
    ↓ проверенный инструмент
Handoff → SubagentRegistry
    ↓ ограниченное поручение
AgentHarness → ToolAction / DelegateAction / Finish
```

`WorkspaceMemory`, события и метрики — готовая инфраструктура вокруг этой цепочки. Они нужны для исполнения и наблюдаемости.

## Основные понятия занятия

| Понятие | Что означает | Для чего нужно |
|---|---|---|
| Навык (`skill`) | Предметная инструкция с описанием, версией, условиями применения и ресурсами. | Подключать только знания, относящиеся к текущей задаче. |
| Ресурс навыка | Отдельный справочный файл рядом с `SKILL.md`. | Не перегружать основную инструкцию подробными правилами и примерами. |
| Дочерний агент (`subagent`) | Отдельный цикл агента с собственным контекстом, инструментами и бюджетом. | Поручать узкую самостоятельную часть задачи без передачи всех полномочий родителя. |
| Agent harness | Код приложения, который проверяет и исполняет предложения модели. | Технически обеспечивать разрешения, бюджеты, повторы, делегирование и наблюдаемость. |
| Реестр (`registry`) | Каталог объектов с правилами уникальности и поиска. | Централизованно находить навыки, инструменты и дочерних агентов и отклонять ошибки настройки. |
| Маршрутизация | Выбор одного владельца запроса по `triggers` и `priority`. | Получать предсказуемый навык или явный конфликт вместо случайного выбора. |
| Список разрешённых инструментов (`allowed_tools`) | Максимальный набор инструментов, доступный конкретному навыку. | Не считать наличие инструмента в приложении автоматическим разрешением на его вызов. |
| JSON Schema и типизированная проверка | Машиночитаемое описание имён и типов аргументов инструмента. | Отклонять пропущенные, лишние и неверно типизированные аргументы до обработчика. |
| Побочный эффект | Изменение состояния: запись файла, изменение внешней системы или запуск операции. | Отличать чтение (`read_only`) от записи в рабочую область (`workspace_write`) и внешнего действия (`external_effect`). |
| Ограниченный повтор (`retry`) | Повтор только заранее разрешённой временной ошибки и только до заданного предела. | Переживать краткий сбой, не скрывая постоянные ошибки и не создавая бесконечный цикл. |
| Поручение (`Handoff`) | Небольшой контракт с целью, контекстом, инструментами и бюджетом дочернего агента. | Передавать ровно необходимые данные и не расширять полномочия при делегировании. |

In [2]:
from dataclasses import dataclass, fields, replace
from pathlib import Path
from tempfile import TemporaryDirectory


class AmbiguousSkillError(RuntimeError):
    """Несколько навыков имеют одинаковое право владеть запросом."""


# Неизменяемая карточка защищает конфигурацию навыка после регистрации.
@dataclass(frozen=True, slots=True)
class Skill:
    """Короткая карточка: когда применять навык и что ему разрешено."""

    name: str
    description: str
    version: str
    triggers: tuple[str, ...]
    priority: int
    allowed_tools: tuple[str, ...]
    subagent: str
    resource: str


# Готовый навык security понадобится для проверки маршрутизации.
security = Skill(
    name="security",
    description="Проверка утечек токенов и проблем доступа.",
    version="2.0.0",
    triggers=("утечк", "токен", "доступ"),
    priority=40,
    allowed_tools=("get_security_facts",),
    subagent="security-researcher",
    resource="references/credential-leak.md",
)

pprint({
    "поля_навыка": [
        "name", "description", "version", "triggers", "priority",
        "allowed_tools", "subagent", "resource",
    ],
    "готовый_пример": security.name,
    "ошибка_маршрутизации": AmbiguousSkillError.__name__,
})

{'готовый_пример': 'security',
 'ошибка_маршрутизации': 'AmbiguousSkillError',
 'поля_навыка': ['name',
                 'description',
                 'version',
                 'triggers',
                 'priority',
                 'allowed_tools',
                 'subagent',
                 'resource']}


Версионируемый `Skill` фиксирует описание, условия срабатывания, `priority`, ресурсы и разрешения; неоднозначная маршрутизация имеет отдельный тип ошибки.

## 1. Карточка навыка

Навык отвечает на пять практических вопросов:

- для каких запросов он подходит;
- какая версия и `priority` участвуют в реестре;
- какие инструменты ему нужны;
- какому дочернему агенту можно делегировать;
- где лежит справочный материал.

In [3]:
def build_billing_skill() -> Skill:
    """Описывает одну предметную область без цикла модели."""
    return Skill(
        name="billing",
        description="Проверка возвратов и вопросов оплаты.",
        version="1.2.0",
        triggers=("возврат", "оплат"),
        priority=20,
        allowed_tools=(
            "online_get_account", "billing_policy", "request_refund", "always_transient",
        ),
        subagent="billing-researcher",
        resource="references/refund-policy.md",
    )


# Проверки ниже задают точный контракт карточки, которую собирает слушатель.
billing = build_billing_skill()
assert billing.name == "billing"
assert billing.version == "1.2.0" and billing.priority == 20
assert "возврат" in billing.triggers and "оплат" in billing.triggers
assert billing.allowed_tools == (
    "online_get_account", "billing_policy", "request_refund", "always_transient",
)
assert billing.subagent == "billing-researcher"
assert billing.resource == "references/refund-policy.md"

pprint({
    "навык_billing": {
        "name": billing.name,
        "version": billing.version,
        "priority": billing.priority,
        "triggers": billing.triggers,
        "allowed_tools": billing.allowed_tools,
        "subagent": billing.subagent,
        "resource": billing.resource,
    }
})

{'навык_billing': {'allowed_tools': ('online_get_account',
                                     'billing_policy',
                                     'request_refund',
                                     'always_transient'),
                   'name': 'billing',
                   'priority': 20,
                   'resource': 'references/refund-policy.md',
                   'subagent': 'billing-researcher',
                   'triggers': ('возврат', 'оплат'),
                   'version': '1.2.0'}}


Навык billing описан декларативно: версия и ресурсы отделены от цикла модели, а список разрешений задаёт максимальный набор возможностей.

### Как навык выглядит на диске

Объект `Skill` выше помогает выбрать подходящие инструкции по тексту запроса. В Deep Agents эти инструкции хранятся в каталоге с файлом `SKILL.md` и дополнительными справочными материалами. Посмотрим на навык billing: в конце занятия мы подключим его к агенту на GigaChat.

In [ ]:
# Навык в Deep Agents — это настоящий каталог с SKILL.md и справочными файлами.
project_root = Path("/content").resolve()
billing_skill_directory = (
    project_root / "skills" / "subagents" / "billing-investigation"
)
billing_skill_file = billing_skill_directory / "SKILL.md"
billing_reference_file = billing_skill_directory / "references" / "refund-policy.md"

skill_source_text = billing_skill_file.read_text(encoding="utf-8")
reference_text = billing_reference_file.read_text(encoding="utf-8")
assert "name: billing-investigation" in skill_source_text
assert "allowed-tools: billing_policy" in skill_source_text
assert "подтвержд" in reference_text

# В выводе показываем только короткий фрагмент, а не всю системную инструкцию.
skill_source_preview = [
    line for line in skill_source_text.splitlines() if line.strip()
][:7]

pprint({
    "каталог_навыка": billing_skill_directory.relative_to(project_root).as_posix(),
    "фрагмент_источника": skill_source_preview,
    "справочный_файл": billing_reference_file.relative_to(project_root).as_posix(),
})

Навык Deep Agents хранится в обычном каталоге: `SKILL.md` задаёт инструкции и инструменты, а справочный файл содержит отдельный материал.

## 2. Выбор навыка

Реестр приводит запрос к одному регистру, собирает все совпадения условий срабатывания и выбирает единственный навык с наибольшим `priority`. Если совпадений нет, возвращается `None`; если у двух победителей равный приоритет, возникает `AmbiguousSkillError`, а не случайный выбор первого.

Такое правило даёт предсказуемый результат и легко проверяется. Позже подходящие навыки можно находить не только по словам, но и по смыслу запроса. При этом окончательный выбор всё равно должен быть единственным и учитывать приоритеты.

In [4]:
class SkillRegistry:
    """Хранит версии навыков и выбирает владельца запроса."""

    def __init__(self, skills: tuple[Skill, ...]):
        """Проверяет уникальность и сохраняет карточки навыков."""
        identities = {(skill.name, skill.version) for skill in skills}
        if len(identities) != len(skills):
            raise ValueError("пара name/version навыка должна быть уникальной")
        self.skills = skills

    def resolve(self, message: str) -> Skill | None:
        """Выбирает владельца с наибольшим приоритетом или сообщает о конфликте."""
        text = message.casefold()
        matches = [
            skill for skill in self.skills
            if any(trigger.casefold() in text for trigger in skill.triggers)
        ]
        if not matches:
            return None
        top_priority = max(skill.priority for skill in matches)
        winners = [skill for skill in matches if skill.priority == top_priority]
        if len(winners) != 1:
            identities = sorted(f"{item.name}@{item.version}" for item in winners)
            raise AmbiguousSkillError(f"конфликт условий срабатывания: {identities}")
        return winners[0]


def choose_skill(message: str, skills: tuple[Skill, ...]) -> Skill | None:
    """Выбирает навык через временный реестр карточек."""
    return SkillRegistry(skills).resolve(message)


# Сценарии покрывают победителя, отсутствие совпадения и ничью приоритетов.
skills = (billing, security)
skill_registry = SkillRegistry(skills)
assert skill_registry.resolve("Нужен возврат по оплате") is billing
assert skill_registry.resolve("Утёк токен доступа") is security
assert skill_registry.resolve("Возврат и доступ к аккаунту") is security
assert skill_registry.resolve("Когда работает поддержка?") is None
try:
    SkillRegistry((billing, replace(billing, name="billing-copy"))).resolve("оплата")
    raise AssertionError("равный приоритет должен быть явным конфликтом")
except AmbiguousSkillError as error:
    _routing_conflict = str(error)

pprint({
    "маршрутизация": {
        message: (chosen.name if (chosen := choose_skill(message, skills)) else None)
        for message in (
            "Нужен возврат по оплате",
            "Утёк токен доступа",
            "Возврат и доступ к аккаунту",
            "Когда работает поддержка?",
        )
    },
    "конфликт_равных_приоритетов": _routing_conflict,
})

{'конфликт_равных_приоритетов': 'конфликт условий срабатывания: '
                                "['billing-copy@1.2.0', 'billing@1.2.0']",
 'маршрутизация': {'Возврат и доступ к аккаунту': 'security',
                   'Когда работает поддержка?': None,
                   'Нужен возврат по оплате': 'billing',
                   'Утёк токен доступа': 'security'}}


`SkillRegistry` выбирает владельца с наибольшим приоритетом, возвращает `None` без совпадения и делает равноправный конфликт явной ошибкой.

### Проверка разрешённого инструмента

Маршрутизация отвечает «кто владеет запросом», а проверка разрешений — «какой инструмент этому навыку доступен». Проверка выполняется обычным Python-кодом до вызова обработчика. Даже зарегистрированный `request_refund` не становится доступным автоматически.

In [5]:
def tool_allowed(skill: Skill, tool_name: str) -> bool:
    """Проверяет карточку разрешений до любого Python-вызова."""
    return tool_name in skill.allowed_tools


# Наличие инструмента в приложении ещё не даёт каждому навыку право его вызвать.
security_tool_checks = {
    "get_security_facts": tool_allowed(security, "get_security_facts"),
    "billing_policy": tool_allowed(security, "billing_policy"),
    "request_refund": tool_allowed(security, "request_refund"),
}
assert security_tool_checks == {
    "get_security_facts": True,
    "billing_policy": False,
    "request_refund": False,
}

pprint({
    "проверки_инструментов_security": security_tool_checks,
    "правило": "зарегистрированный инструмент не равен разрешённому инструменту",
})

{'правило': 'зарегистрированный инструмент не равен разрешённому инструменту',
 'проверки_инструментов_security': {'billing_policy': False,
                                    'get_security_facts': True,
                                    'request_refund': False}}


Проверка разрешений отделяет наличие инструмента в приложении от права конкретного навыка: security не получает инструмент billing или инструмент выполнения возврата.

### Настоящие инструменты LangChain

Строки в `allowed_tools` должны ссылаться на настоящие инструменты приложения. Создадим пять инструментов `@tool`: три прикладных инструмента чтения, инструмент внешней операции и учебный инструмент с временной ошибкой. Затем посмотрим их схемы JSON и локально вызовем инструмент безопасности без участия модели.

In [6]:
# Декоратор @tool строит описание инструмента из сигнатуры Python-функции.
from langchain.tools import tool


# Первые три инструмента только читают данные и не меняют внешние системы.
@tool("get_security_facts")
def preview_security_facts(topic: str) -> dict:
    """Прочитать проверенный факт о безопасности без побочного эффекта."""
    return {"topic": topic, "status": "review_required"}


@tool("billing_policy")
def preview_billing_policy(query: str) -> str:
    """Прочитать учебное правило возврата без выполнения операции."""
    return f"Для {query} нужно подтверждение оператора."


@tool("online_get_account")
def preview_online_get_account(account_id: str) -> dict:
    """Прочитать тариф аккаунта без побочного эффекта."""
    return {"account_id": account_id, "plan": "бизнес", "status": "active"}


# Счётчик доказывает, что внешний эффект не был выполнен случайно.
_external_effect_calls: list[dict] = []

@tool("request_refund")
def preview_request_refund(payment_id: str, amount: int) -> dict:
    """Выполнить возврат; контур не должен вызывать его напрямую."""
    _external_effect_calls.append({"payment_id": payment_id, "amount": amount})
    return {"status": "scheduled"}


# Этот предсказуемый сбой понадобится для проверки ограниченных повторов.
_transient_attempts: list[str] = []

@tool("always_transient")
def preview_always_transient(query: str) -> str:
    """Учебный инструмент только для чтения, всегда дающий временный сбой."""
    _transient_attempts.append(query)
    raise TimeoutError("временное превышение ожидания поставщика")


# LangChain строит схему JSON отдельно для каждого зарегистрированного инструмента.
preview_tools = (
    preview_security_facts, preview_billing_policy, preview_online_get_account,
    preview_request_refund, preview_always_transient,
)
preview_tool_schemas = {
    name: item.args_schema.model_json_schema()["properties"]
    for item in preview_tools
    for name in (item.name,)
}
security_fact = preview_security_facts.invoke({"topic": "token"})
assert set(preview_tool_schemas["get_security_facts"]) == {"topic"}
assert security_fact["status"] == "review_required"
assert _external_effect_calls == []

pprint({
    "схемы_инструментов": preview_tool_schemas,
    "локальный_вызов": security_fact,
    "вызовы_внешнего_эффекта": list(_external_effect_calls),
})

{'вызовы_внешнего_эффекта': [],
 'локальный_вызов': {'status': 'review_required', 'topic': 'token'},
 'схемы_инструментов': {'always_transient': {'query': {'title': 'Query',
                                                       'type': 'string'}},
                        'billing_policy': {'query': {'title': 'Query',
                                                     'type': 'string'}},
                        'get_security_facts': {'topic': {'title': 'Topic',
                                                         'type': 'string'}},
                        'online_get_account': {'account_id': {'title': 'Account '
                                                                       'Id',
                                                              'type': 'string'}},
                        'request_refund': {'amount': {'title': 'Amount',
                                                      'type': 'integer'},
                                           'payment_id': {'title': 

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool

convert_to_openai_tool(preview_online_get_account)

{'type': 'function',
 'function': {'name': 'online_get_account',
  'description': 'Прочитать тариф аккаунта без побочного эффекта.',
  'parameters': {'properties': {'account_id': {'type': 'string'}},
   'required': ['account_id'],
   'type': 'object'}}}

LangChain `@tool` превращает Python-функцию в схему и обработчик; исполнитель внешней операции существует в приложении, но контур его не вызывает.

### Реестр инструментов и сведения о побочных эффектах

Каждый инструмент, созданный через `@tool`, уже содержит имя, схему аргументов и Python-обработчик. Дополнительно наш контур хранит правила его выполнения: только чтение (`read_only`), запись в рабочую область (`workspace_write`) или внешний эффект (`external_effect`), а также перечень временных ошибок, после которых разрешён повтор.

Например, `billing_policy` только читает данные, `request_refund` описывает внешний эффект, а `online_get_account` можно повторить после `TimeoutError`. `ToolRegistry` собирает эти сведения для всех инструментов и отклоняет повторяющиеся имена и неизвестные виды эффектов.

In [8]:
@dataclass(frozen=True, slots=True)
class ToolSpec:
    """Служебные сведения исполняющей среды рядом с инструментом LangChain."""

    tool: object
    effect: str
    retry_on: tuple[type[BaseException], ...] = ()


# Реестр проверяет правила среды один раз, до выполнения плана агента.
class ToolRegistry:
    """Хранит инструменты вместе с правилами их выполнения."""

    VALID_EFFECTS = {"read_only", "workspace_write", "external_effect"}

    def __init__(self, specs: tuple[ToolSpec, ...]):
        """Проверяет и регистрирует набор описаний инструментов."""
        self._specs: dict[str, ToolSpec] = {}
        for spec in specs:
            name = spec.tool.name
            if name in self._specs:
                raise ValueError(f"инструмент уже зарегистрирован: {name}")
            if spec.effect not in self.VALID_EFFECTS:
                raise ValueError(f"неизвестный вид побочного эффекта: {spec.effect}")
            self._specs[name] = spec

    def get(self, name: str) -> ToolSpec | None:
        """Возвращает описание инструмента по имени, если оно известно."""
        return self._specs.get(name)

    @property
    def names(self) -> frozenset[str]:
        """Возвращает неизменяемый набор зарегистрированных имён."""
        return frozenset(self._specs)


# Повтор разрешён только для явно перечисленных временных ошибок.
preview_tool_registry = ToolRegistry((
    ToolSpec(preview_security_facts, "read_only"),
    ToolSpec(preview_billing_policy, "read_only"),
    ToolSpec(preview_online_get_account, "read_only", (TimeoutError,)),
    ToolSpec(preview_request_refund, "external_effect"),
    ToolSpec(preview_always_transient, "read_only", (TimeoutError,)),
))
assert preview_tool_registry.names == {
    "get_security_facts", "billing_policy", "online_get_account",
    "request_refund", "always_transient",
}

pprint({
    "реестр_инструментов": {
        name: {
            "effect": preview_tool_registry.get(name).effect,
            "retry_on": [item.__name__ for item in preview_tool_registry.get(name).retry_on],
        }
        for name in sorted(preview_tool_registry.names)
    }
})

{'реестр_инструментов': {'always_transient': {'effect': 'read_only',
                                              'retry_on': ['TimeoutError']},
                         'billing_policy': {'effect': 'read_only',
                                            'retry_on': []},
                         'get_security_facts': {'effect': 'read_only',
                                                'retry_on': []},
                         'online_get_account': {'effect': 'read_only',
                                                'retry_on': ['TimeoutError']},
                         'request_refund': {'effect': 'external_effect',
                                            'retry_on': []}}}


`ToolRegistry` добавляет к схеме вид побочного эффекта и правила повторов: эти границы исполняющей среды нельзя выразить одним описанием инструмента.

## 3. Проверка типов перед обработчиком

Одной проверки имени недостаточно. Исполняющая среда должна убедиться, что инструмент разрешён выбранному навыку, зарегистрирован в каталоге и получил ровно те аргументы, которые описаны его схемой JSON.

Реализуйте проверку, которая возвращает `ValidatedToolRequest`, но не вызывает обработчик. Отрицательные сценарии отдельно проверяют запрещённый инструмент, пропущенный аргумент и лишнее поле.

In [9]:
# После проверки цикл получает единый объект вместо разрозненных данных.
@dataclass(frozen=True, slots=True)
class ValidatedToolRequest:
    """Инструмент и нормализованные аргументы после проверки среды."""

    tool: object
    arguments: dict[str, object]
    effect: str
    retry_on: tuple[type[BaseException], ...]

pprint({
    "поля_проверенного_запроса": ["tool", "arguments", "effect", "retry_on"],
    "порядок_проверок": ["разрешения навыка", "каталог инструментов", "точная схема JSON"],
})

{'поля_проверенного_запроса': ['tool', 'arguments', 'effect', 'retry_on'],
 'порядок_проверок': ['разрешения навыка',
                      'каталог инструментов',
                      'точная схема JSON']}


`ValidatedToolRequest` переносит нормализованные аргументы и сведения об эффектах и повторах; проверка остаётся отделена от вызова обработчика.

In [10]:
def validate_tool_request(
    skill: Skill, tool_name: str, arguments: dict, tool_registry: ToolRegistry,
) -> ValidatedToolRequest:
    """Проверяет разрешения и схему до вызова обработчика."""
    if tool_name not in skill.allowed_tools:
        raise ValueError(f"навык {skill.name} не разрешает {tool_name}")
    spec = tool_registry.get(tool_name)
    if spec is None:
        raise ValueError(f"инструмент отсутствует в каталоге: {tool_name}")
    properties = spec.tool.args_schema.model_json_schema()["properties"]
    if not isinstance(arguments, dict) or set(arguments) != set(properties):
        raise ValueError("аргументы не совпадают со схемой")
    normalized = spec.tool.args_schema.model_validate(arguments).model_dump()
    return ValidatedToolRequest(
        tool=spec.tool, arguments=normalized,
        effect=spec.effect, retry_on=spec.retry_on,
    )


# Положительный сценарий возвращает данные, но ещё не вызывает инструмент.
validated_policy = validate_tool_request(
    billing, "billing_policy", {"query": "возврат A-1"}, preview_tool_registry,
)
assert validated_policy.tool.name == "billing_policy"
assert validated_policy.arguments == {"query": "возврат A-1"}

# Три отрицательных сценария проверяют разные границы одного контракта.
_tool_request_rejections = set()
for label, tool_name, arguments in (
    ("запрещённый", "get_security_facts", {"topic": "token"}),
    ("пропущенный", "billing_policy", {}),
    ("лишний", "billing_policy", {"query": "возврат", "amount": 100}),
):
    try:
        validate_tool_request(billing, tool_name, arguments, preview_tool_registry)
        raise AssertionError("недопустимый запрос к инструменту должен быть отклонён")
    except ValueError:
        _tool_request_rejections.add(label)
assert _tool_request_rejections == {"запрещённый", "пропущенный", "лишний"}

pprint({
    "проверенный_инструмент": validated_policy.tool.name,
    "нормализованные_аргументы": validated_policy.arguments,
    "побочный_эффект": validated_policy.effect,
    "отклонённые_запросы": sorted(_tool_request_rejections),
    "обработчик_вызван_проверкой": False,
})

{'нормализованные_аргументы': {'query': 'возврат A-1'},
 'обработчик_вызван_проверкой': False,
 'отклонённые_запросы': ['запрещённый', 'лишний', 'пропущенный'],
 'побочный_эффект': 'read_only',
 'проверенный_инструмент': 'billing_policy'}


Запрос к правилу billing прошёл три проверки, а запрещённый, неполный и избыточный варианты отклонены до обработчика; сама проверка не создаёт побочный эффект.

### Из списка разрешений в конфигурацию дочернего агента

Следующая небольшая функция берёт имена из навыка, находит настоящие объекты инструментов и добавляет отдельный бюджет. Так видно, откуда берётся узкий набор возможностей дочернего агента до вызова `create_deep_agent`.

In [11]:
def build_subagent_preview(skill: Skill, tool_registry: ToolRegistry) -> dict:
    """Связывает разрешения навыка с настоящими инструментами LangChain."""
    # Неизвестное имя инструмента означает ошибку настройки, а не ошибку модели.
    missing = set(skill.allowed_tools) - tool_registry.names
    if missing:
        raise ValueError(f"инструменты отсутствуют в каталоге: {sorted(missing)}")
    return {
        "name": skill.subagent,
        "skill": skill.name,
        "tools": tuple(tool_registry.get(name).tool for name in skill.allowed_tools),
        "limits": {"model_calls": 3, "tool_calls": 4},
    }


# Дочерний агент security получает только свой инструмент чтения.
security_subagent_preview = build_subagent_preview(security, preview_tool_registry)
assert security_subagent_preview["name"] == "security-researcher"
assert [item.name for item in security_subagent_preview["tools"]] == [
    "get_security_facts",
]
assert "billing_policy" not in {
    item.name for item in security_subagent_preview["tools"]
}

pprint({
    "дочерний_агент": {
        "name": security_subagent_preview["name"],
        "skill": security_subagent_preview["skill"],
        "tools": [item.name for item in security_subagent_preview["tools"]],
        "limits": security_subagent_preview["limits"],
    }
})

{'дочерний_агент': {'limits': {'model_calls': 3, 'tool_calls': 4},
                    'name': 'security-researcher',
                    'skill': 'security',
                    'tools': ['get_security_facts']}}


Конфигурация дочернего агента получает настоящие объекты инструментов только через разрешения навыка и добавляет отдельные бюджеты до сборки графа.

## 4. Ограниченное поручение (`Handoff`)

Делегирование — не пересылка всей истории. Управляющий агент формирует новый небольшой контракт:

- короткая цель;
- только нужный `account_id`;
- один разрешённый инструмент чтения правил;
- отдельный лимит вызовов модели.

Это проще отлаживать и безопаснее, чем отдавать дочернему агенту весь контекст и все инструменты родителя.

In [12]:
# Поручение не переносит исходную историю диалога целиком.
@dataclass(frozen=True, slots=True)
class Handoff:
    """Минимальные данные, которые управляющий агент передаёт дочернему."""

    subagent: str
    goal: str
    context: dict[str, str]
    allowed_tools: tuple[str, ...]
    model_call_limit: int

pprint({
    "поля_поручения": [
        "subagent", "goal", "context", "allowed_tools", "model_call_limit",
    ]
})

{'поля_поручения': ['subagent',
                    'goal',
                    'context',
                    'allowed_tools',
                    'model_call_limit']}


`Handoff` отделён от полной истории диалога и содержит только цель, контекст, инструменты и собственный бюджет дочернего агента.

In [13]:
def prepare_handoff(skill: Skill, account_id: str) -> Handoff:
    """Сужает контекст и полномочия перед делегированием."""
    if "billing_policy" not in skill.allowed_tools:
        raise ValueError("навык не разрешает billing_policy")
    return Handoff(
        subagent=skill.subagent,
        goal="Проверь правило возврата и верни только подтверждённый факт.",
        context={"account_id": account_id},
        allowed_tools=("billing_policy",),
        model_call_limit=3,
    )


# Проверки фиксируют принцип минимально необходимых данных и полномочий.
handoff = prepare_handoff(billing, "A-1")
assert handoff.subagent == "billing-researcher"
assert handoff.context == {"account_id": "A-1"}
assert handoff.allowed_tools == ("billing_policy",)
assert set(handoff.allowed_tools) <= set(billing.allowed_tools)
assert handoff.model_call_limit == 3

pprint({
    "поручение": {
        "subagent": handoff.subagent,
        "goal": handoff.goal,
        "context": handoff.context,
        "allowed_tools": handoff.allowed_tools,
        "model_call_limit": handoff.model_call_limit,
    }
})

{'поручение': {'allowed_tools': ('billing_policy',),
               'context': {'account_id': 'A-1'},
               'goal': 'Проверь правило возврата и верни только подтверждённый '
                       'факт.',
               'model_call_limit': 3,
               'subagent': 'billing-researcher'}}


Дочерний агент billing получает один `account_id` и один инструмент чтения; делегирование не расширяет разрешения исходного навыка.

### Результат дочернего агента тоже проверяется

Узкое поручение ограничивает вход, но родитель не должен автоматически доверять выходу. Готовый контракт ниже проверяет отправителя, конечное состояние и наличие фактов с источником перед использованием результата в итоговом ответе.

In [14]:
# Родитель принимает не свободный текст, а небольшой проверяемый контракт.
@dataclass(frozen=True, slots=True)
class SubagentResult:
    """Минимальный проверяемый ответ дочернего агента."""

    owner: str
    facts: tuple[str, ...]
    source: str
    status: str


def accept_subagent_result(result: SubagentResult, expected_owner: str) -> SubagentResult:
    """Родитель проверяет отправителя, состояние и наличие подтверждений."""
    if result.owner != expected_owner:
        raise ValueError("ответ пришёл от неожиданного дочернего агента")
    if result.status != "success" or not result.facts or not result.source:
        raise ValueError("результат дочернего агента не содержит подтверждённых данных")
    return result


# Сначала проверяем допустимый результат, затем — отказ при неверном отправителе.
accepted_subagent_result = accept_subagent_result(
    SubagentResult(
        owner="billing-researcher",
        facts=("для возврата нужно подтверждение оператора",),
        source="references/refund-policy.md",
        status="success",
    ),
    expected_owner="billing-researcher",
)
try:
    accept_subagent_result(
        SubagentResult("unknown", ("ok",), "unknown.md", "success"),
        expected_owner="billing-researcher",
    )
    raise AssertionError("неверный отправитель должен быть отклонён")
except ValueError:
    _wrong_subagent_owner_rejected = True

pprint({
    "принятый_результат": {
        "owner": accepted_subagent_result.owner,
        "facts": accepted_subagent_result.facts,
        "source": accepted_subagent_result.source,
        "status": accepted_subagent_result.status,
    },
    "неверный_отправитель_отклонён": _wrong_subagent_owner_rejected,
})

{'неверный_отправитель_отклонён': True,
 'принятый_результат': {'facts': ('для возврата нужно подтверждение '
                                  'оператора',),
                        'owner': 'billing-researcher',
                        'source': 'references/refund-policy.md',
                        'status': 'success'}}


Родитель проверяет не только текст ответа: ожидаемый отправитель, успешное состояние, факты и источник образуют минимальный контракт результата.

## 5. Готовый пример: локальный проход управляющего агента

Готовая функция связывает уже знакомые части без модели: маршрутизация выбирает навык, исполняющая среда проверяет `online_get_account`, Python вызывает обработчик только для чтения, затем управляющий агент создаёт ограниченное поручение.

Запустите ячейку и прочитайте `run_local_support` сверху вниз. Короткий `trace` фиксирует порядок, а запрос с неизвестной предметной областью завершается как `no_skill` до инструмента.

In [15]:
def run_local_support(message: str, account_id: str) -> dict:
    """Выполняет короткий детерминированный путь без вызова модели."""
    selected = choose_skill(message, skills)
    if selected is None:
        return {"status": "no_skill", "trace": ("skill:none",)}
    account_request = validate_tool_request(
        selected, "online_get_account", {"account_id": account_id},
        preview_tool_registry,
    )
    account = account_request.tool.invoke(account_request.arguments)
    delegated = prepare_handoff(selected, account_id)
    return {
        "status": "delegated",
        "skill": selected.name,
        "account": account,
        "handoff": delegated,
        "trace": (
            f"skill:{selected.name}",
            f"tool:{account_request.tool.name}",
            f"handoff:{delegated.subagent}",
        ),
    }


# Сравниваем успешную маршрутизацию с запросом без подходящего навыка.
local_success = run_local_support("Проверь возврат по оплате", "A-1")
local_no_skill = run_local_support("Когда работает поддержка?", "A-1")
assert local_success["status"] == "delegated"
assert local_success["skill"] == "billing"
assert local_success["account"]["plan"] == "бизнес"
assert local_success["handoff"].context == {"account_id": "A-1"}
assert local_success["trace"] == (
    "skill:billing", "tool:online_get_account", "handoff:billing-researcher",
)
assert local_no_skill == {"status": "no_skill", "trace": ("skill:none",)}

pprint({
    "локальный_управляющий_агент": {
        "status": local_success["status"],
        "skill": local_success["skill"],
        "account": local_success["account"],
        "handoff": {
            "subagent": local_success["handoff"].subagent,
            "context": local_success["handoff"].context,
            "tools": local_success["handoff"].allowed_tools,
        },
        "trace": local_success["trace"],
    },
    "неизвестная_область": local_no_skill,
})

{'локальный_управляющий_агент': {'account': {'account_id': 'A-1',
                                             'plan': 'бизнес',
                                             'status': 'active'},
                                 'handoff': {'context': {'account_id': 'A-1'},
                                             'subagent': 'billing-researcher',
                                             'tools': ('billing_policy',)},
                                 'skill': 'billing',
                                 'status': 'delegated',
                                 'trace': ('skill:billing',
                                           'tool:online_get_account',
                                           'handoff:billing-researcher')},
 'неизвестная_область': {'status': 'no_skill', 'trace': ('skill:none',)}}


Локальный проход управляющего агента связывает маршрутизацию, типизированную проверку, чтение и поручение в наблюдаемый `trace`; неизвестная область завершается без инструмента.

## 6. Готовая инфраструктура: ограниченная рабочая память

Deep Agents использует файлы для планов и заметок. Локальный `WorkspaceMemory` показывает ту же границу: разрешены только относительные пути внутри одного корневого каталога и ограниченный размер записи.

Обратите внимание на три шага готового `resolve()`:

1. отклонить пустой или абсолютный путь;
2. получить итоговый путь через `(root / relative).resolve()`;
3. доказать принадлежность корню через `candidate.relative_to(root)`.

`relative_to` поднимает `ValueError`, если итоговый путь вышел наружу. Это ограничение путей файловых инструментов, а не изоляция Python-процесса или сети.

In [16]:
class WorkspaceMemory:
    """Рабочая память в пределах каталога; не изоляция процесса или сети."""

    def __init__(self, root: Path, max_bytes: int = 4096):
        """Создаёт рабочую область и задаёт предел размера записи."""
        self.root = root.resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.max_bytes = max_bytes

    def resolve(self, relative: str) -> Path:
        """Преобразует безопасный относительный путь в абсолютный."""
        if not isinstance(relative, str) or not relative.strip() or relative == ".":
            raise ValueError("нужен непустой относительный путь")
        relative_path = Path(relative)
        if relative_path.is_absolute():
            raise ValueError("абсолютный путь запрещён")
        candidate = (self.root / relative_path).resolve()
        try:
            candidate.relative_to(self.root)
        except ValueError as error:
            raise ValueError("путь выходит за пределы рабочей области") from error
        return candidate

    def write_text(self, relative: str, content: str) -> str:
        """Записывает небольшой текстовый файл внутри рабочей области."""
        if len(content.encode("utf-8")) > self.max_bytes:
            raise ValueError("файл рабочей области превышает max_bytes")
        target = self.resolve(relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return target.relative_to(self.root).as_posix()

    def read_text(self, relative: str) -> str:
        """Читает текстовый файл после проверки его пути."""
        return self.resolve(relative).read_text(encoding="utf-8")


# Временный каталог изолирует учебные файлы от проекта слушателя.
_workspace_tmp = TemporaryDirectory(prefix="aia-lab3-")
workspace = WorkspaceMemory(Path(_workspace_tmp.name))
assert workspace.write_text("plans/support.md", "1. аккаунт\n2. правило") == "plans/support.md"
assert workspace.read_text("plans/support.md").startswith("1. аккаунт")
# Отрицательные сценарии проверяют выход наружу и превышение размера.
_blocked_workspace_cases = set()
for label, unsafe in (("выход", "../secret.txt"), ("абсолютный", "/etc/passwd")):
    try:
        workspace.read_text(unsafe)
        raise AssertionError("опасный путь должен быть отклонён")
    except ValueError:
        _blocked_workspace_cases.add(label)
try:
    workspace.write_text("plans/huge.txt", "x" * 5000)
    raise AssertionError("слишком большая запись должна быть отклонена")
except ValueError:
    _oversized_workspace_write_rejected = True

pprint({
    "рабочая_область": {
        "безопасный_файл": "plans/support.md",
        "содержимое": workspace.read_text("plans/support.md").splitlines(),
        "заблокировано": sorted(_blocked_workspace_cases),
        "большая_запись_отклонена": _oversized_workspace_write_rejected,
        "область_ограничения": "ограничение путей, а не изоляция процесса",
    }
})

{'рабочая_область': {'безопасный_файл': 'plans/support.md',
                     'большая_запись_отклонена': True,
                     'заблокировано': ['абсолютный', 'выход'],
                     'область_ограничения': 'ограничение путей, а не изоляция '
                                            'процесса',
                     'содержимое': ['1. аккаунт', '2. правило']}}


`WorkspaceMemory` допускает ограниченную запись внутри корневого каталога и блокирует выход наружу и абсолютные пути; это ограничение файловых инструментов, а не изоляция процесса или сети.

## 7. Пятое задание: контур выполнения (`AgentHarness`)

Последнее локальное задание связывает все части:

`реестр навыков → бюджет шагов → проверка инструмента → повтор или побочный эффект → дочерний агент → завершение`

`AgentHarness` получает через конструктор реестры навыков, инструментов и дочерних агентов, рабочую область и обработчики событий. Эти компоненты можно заменять или дополнять, не изменяя основной цикл выполнения. Бюджет шагов и предел повторов задаются отдельно при вызове `run()`.

`TraceRecorder` сохраняет безопасные события жизненного цикла без исходных аргументов и вывода, а `MetricsHook` считает те же события, не изменяя выполнение.

Один план проверяет успешный путь; ещё три — исчерпание бюджета шагов, предложение операции вместо внешнего эффекта и исчерпание повторов в пределах того же шага.

Все возможные причины завершения известны заранее:

| Причина | Когда возникает |
|---|---|
| `no_skill` | запрос не относится ни к одному навыку |
| `step_budget_exhausted` | перед следующим действием закончились шаги |
| `approval_required` | инструмент описывает внешний эффект |
| `tool_retry_exhausted` | временная ошибка не исчезла в пределах повторов |
| `invalid_action` | план содержит неизвестный тип действия |
| `no_final_answer` | план закончился без `Finish` |
| `final_answer` | выполнен `Finish` |

### Понятия локального контура

| Понятие | Что означает | Для чего нужно |
|---|---|---|
| Шаг и `StepBudget` | Один занятый слот на действие плана и счётчик доступных слотов. | Остановить выполнение до лишнего действия; повтор инструмента остаётся внутри уже занятого шага. |
| Действие плана | Типизированная команда: `ToolAction`, `DelegateAction` или `Finish`. | Сделать ветвление явным и проверяемым обычным Python-кодом. |
| Наблюдение (`Observation`) | Проверенный результат инструмента только для чтения. | Передать факты следующему шагу без повторного вызова инструмента. |
| `EffectProposal` | Описание внешней операции, которая ещё не выполнена. | Вынести подтверждение и исполнение опасного действия за пределы автоматического цикла. |
| Результат дочернего агента (`SubagentResult`) | Факты, источник, состояние и имя отправителя. | Проверить ответ дочернего агента до использования родителем. |
| Обработчик события (`hook`) | Подключаемая функция, которая получает `TraceEvent`. | Добавлять наблюдение за циклом без изменения его ветвей. |
| `Trace` / `tracing` | Последовательность безопасных событий выполнения. | Понять фактический путь агента, не сохраняя скрытые рассуждения, аргументы и результаты инструментов. |
| Метрика | Числовой показатель, например количество повторов или вызовов. | Сравнивать запуски и замечать исчерпание бюджета или рост ошибок. |

### События и бюджет шага

Сначала определим то, что не зависит от ветвей плана: безопасное событие `TraceEvent`, счётчик шагов и два подключаемых обработчика. Обработчики получают одно и то же событие, но один сохраняет `trace`, а второй считает метрики.

In [18]:
@dataclass(frozen=True, slots=True)
class TraceEvent:
    """Безопасное событие без аргументов и результатов инструментов."""

    event: str
    step: int
    subject: str | None = None
    status: str | None = None
    attempt: int | None = None


class StepBudget:
    """Ограничивает число действий одного запуска контура."""

    def __init__(self, max_steps: int):
        """Создаёт счётчик с неотрицательным пределом шагов."""
        if type(max_steps) is not int or max_steps < 0:
            raise ValueError("max_steps должен быть неотрицательным целым числом")
        self.max_steps = max_steps
        self.used = 0

    def take(self) -> int | None:
        """Занимает следующий шаг или возвращает None при исчерпании."""
        if self.used >= self.max_steps:
            return None
        self.used += 1
        return self.used


class TraceRecorder:
    """Сохраняет безопасные события trace в памяти."""

    def __init__(self):
        """Создаёт пустую последовательность событий."""
        self.events: list[TraceEvent] = []

    def __call__(self, event: TraceEvent) -> None:
        """Добавляет одно событие без изменения цикла."""
        self.events.append(event)


class MetricsHook:
    """Подсчитывает события контура по их именам."""

    def __init__(self):
        """Создаёт пустой набор счётчиков."""
        self.counts: dict[str, int] = {}

    def __call__(self, event: TraceEvent) -> None:
        """Увеличивает счётчик полученного события."""
        self.counts[event.event] = self.counts.get(event.event, 0) + 1

pprint({
    "наблюдаемость": {
        "бюджет": StepBudget.__name__,
        "обработчики": [TraceRecorder.__name__, MetricsHook.__name__],
        "trace_fields": [field.name for field in fields(TraceEvent)],
    }
})

{'наблюдаемость': {'trace_fields': ['event',
                                    'step',
                                    'subject',
                                    'status',
                                    'attempt'],
                   'бюджет': 'StepBudget',
                   'обработчики': ['TraceRecorder', 'MetricsHook']}}


`StepBudget` ограничивает действия, а `TraceRecorder` и `MetricsHook` независимо наблюдают одинаковые безопасные события.

### Четыре типа результата плана

Тип действия делает ветвление явным. `ToolAction` запрашивает инструмент, `DelegateAction` передаёт поручение, `Finish` завершает план, а `EffectProposal` описывает внешний эффект, который ещё не выполнен.

In [19]:
@dataclass(frozen=True, slots=True)
class EffectProposal:
    """Предложение внешнего эффекта без его выполнения."""

    skill: str
    tool: str
    arguments: dict[str, object]


@dataclass(frozen=True, slots=True)
class ToolAction:
    """Запрос плана к именованному инструменту."""

    name: str
    arguments: dict[str, object]


@dataclass(frozen=True, slots=True)
class DelegateAction:
    """Поручение, которое нужно передать дочернему агенту."""

    handoff: Handoff


@dataclass(frozen=True, slots=True)
class Finish:
    """Явное завершение плана с итоговым ответом."""

    answer: str

pprint({
    "типы_действий": [ToolAction.__name__, DelegateAction.__name__, Finish.__name__],
    "внешний_эффект": EffectProposal.__name__,
})

{'внешний_эффект': 'EffectProposal',
 'типы_действий': ['ToolAction', 'DelegateAction', 'Finish']}


Типы действий отделяют запрос инструмента, поручение и завершение; `EffectProposal` хранит внешний эффект, не выполняя его.

### Реестр дочерних агентов

Реестр проверяет, что выбранный навык разрешает дочернего агента и его инструменты. Только после этих проверок он вызывает обработчик и проверяет `SubagentResult`.

In [20]:
class SubagentRegistry:
    """Регистрирует и безопасно запускает дочерние обработчики."""

    def __init__(self):
        """Создаёт пустой реестр дочерних обработчиков."""
        self._handlers: dict[str, object] = {}

    def register(self, name: str, handler) -> None:
        """Связывает уникальное имя дочернего агента с обработчиком."""
        if name in self._handlers:
            raise ValueError(f"дочерний агент уже зарегистрирован: {name}")
        self._handlers[name] = handler

    def launch(
        self, skill: Skill, handoff: Handoff, memory: WorkspaceMemory,
    ) -> SubagentResult:
        """Проверяет поручение, запускает обработчик и проверяет результат."""
        if handoff.subagent != skill.subagent:
            raise ValueError("навык не разрешает этого дочернего агента")
        if not set(handoff.allowed_tools) <= set(skill.allowed_tools):
            raise ValueError("поручение расширяет список разрешённых инструментов")
        handler = self._handlers.get(handoff.subagent)
        if handler is None:
            raise ValueError("дочерний агент не зарегистрирован")
        raw_result = handler(handoff, memory)
        return accept_subagent_result(raw_result, expected_owner=handoff.subagent)


def _billing_worker(handoff: Handoff, memory: WorkspaceMemory) -> SubagentResult:
    """Возвращает учебный факт billing и сохраняет его в рабочей области."""
    note = "для возврата нужно подтверждение оператора"
    memory.write_text("subagents/billing-result.md", note)
    return SubagentResult(
        owner=handoff.subagent,
        facts=(note,),
        source="references/refund-policy.md",
        status="success",
    )


subagent_registry = SubagentRegistry()
subagent_registry.register("billing-researcher", _billing_worker)

pprint({
    "реестр_дочерних_агентов": sorted(subagent_registry._handlers),
    "контракт_результата": [field.name for field in fields(SubagentResult)],
})

{'контракт_результата': ['owner', 'facts', 'source', 'status'],
 'реестр_дочерних_агентов': ['billing-researcher']}


`SubagentRegistry` проверяет имя, разрешения и контракт результата до возвращения фактов управляющему агенту.

### Ветвь инструмента

Вызов инструмента вынесен из основного цикла. Здесь сосредоточены проверка схемы, ограниченный повтор временной ошибки и превращение внешнего эффекта в `EffectProposal`.

In [25]:
@dataclass(frozen=True, slots=True)
class ToolBranchResult:
    """Один из трёх исходов ветви инструмента."""

    observation: dict | None = None
    proposal: EffectProposal | None = None
    reason: str | None = None


def execute_tool_action(
    harness, skill: Skill, action: ToolAction, step: int,
    max_tool_attempts: int,
) -> ToolBranchResult:
    """Проверяет и выполняет одну ветвь инструмента внутри занятого шага."""
    request = validate_tool_request(
        skill, action.name, action.arguments, harness.tools,
    )
    harness.emit("tool.started", step, subject=action.name)
    if request.effect == "external_effect":
        proposal = EffectProposal(skill.name, action.name, request.arguments)
        harness.emit("effect.proposed", step, subject=action.name)
        return ToolBranchResult(proposal=proposal, reason="approval_required")

    for attempt in range(1, max_tool_attempts + 1):
        harness.emit(
            "tool.attempt", step, subject=action.name, attempt=attempt,
        )
        try:
            output = request.tool.invoke(request.arguments)
        except request.retry_on:
            if attempt == max_tool_attempts:
                return ToolBranchResult(reason="tool_retry_exhausted")
            harness.emit(
                "tool.retry", step, subject=action.name, attempt=attempt,
            )
        else:
            harness.emit("tool.finished", step, subject=action.name)
            return ToolBranchResult(
                observation={"tool": action.name, "output": output},
            )
    raise RuntimeError("ветвь инструмента завершилась без результата")

pprint({
    "ветвь_инструмента": execute_tool_action.__name__,
    "исходы": ["observation", "approval_required", "tool_retry_exhausted"],
})

{'ветвь_инструмента': 'execute_tool_action',
 'исходы': ['observation', 'approval_required', 'tool_retry_exhausted']}


Ветвь инструмента проверяет запрос, оставляет повтор внутри одного шага и не вызывает обработчик внешнего эффекта.

### Оболочка `AgentHarness`

Оболочка хранит зависимости и публикует события. Сам порядок выполнения остаётся отдельной функцией задания, поэтому его можно читать без кода реестров и обработчиков.

In [26]:
class AgentHarness:
    """Связывает реестры, бюджеты и обработчики в один цикл."""

    def __init__(
        self, *, skills: SkillRegistry, tools: ToolRegistry,
        subagents: SubagentRegistry, memory: WorkspaceMemory, hooks=(),
    ):
        """Сохраняет зависимости контура и подключаемые обработчики."""
        self.skills = skills
        self.tools = tools
        self.subagents = subagents
        self.memory = memory
        self.hooks = tuple(hooks)

    def emit(self, event: str, step: int, **safe_fields) -> None:
        """Передаёт безопасное событие всем подключённым обработчикам."""
        trace_event = TraceEvent(event=event, step=step, **safe_fields)
        for hook in self.hooks:
            hook(trace_event)

    def run(
        self, message: str, plan: tuple, *, max_steps: int,
        max_tool_attempts: int = 2,
    ) -> dict:
        """Передаёт выполнение плана функции учебного задания."""
        return run_agent_harness(
            self, message, plan, max_steps=max_steps,
            max_tool_attempts=max_tool_attempts,
        )


def _harness_result(
    reason, skill, budget, observations, results, proposals, answer=None,
):
    """Собирает единый итог для всех ветвей завершения контура."""
    return {
        "reason": reason,
        "skill": skill.name if skill else None,
        "steps_used": budget.used,
        "observations": tuple(observations),
        "subagent_results": tuple(results),
        "pending_effects": tuple(proposals),
        "answer": answer,
    }

pprint({
    "контур": AgentHarness.__name__,
    "методы": ["emit", "run"],
    "единый_результат": _harness_result.__name__,
})

{'единый_результат': '_harness_result',
 'контур': 'AgentHarness',
 'методы': ['emit', 'run']}


`AgentHarness` хранит зависимости и события, а учебная функция отдельно задаёт порядок выполнения плана.

### Задание: связать ветви в один цикл

Каркас уже выбирает навык, создаёт бюджет, публикует начальные и конечные события, занимает шаг до действия и содержит три готовых `isinstance`-ветви. Заполните только их тела:

```text
ToolAction     → execute_tool_action → observation / proposal / reason
DelegateAction → emit started → subagents.launch → emit finished
Finish         → finish("final_answer", action.answer)
```

Неизвестное действие уже возвращает `invalid_action`, а план без `Finish` — `no_final_answer`.

In [28]:
def run_agent_harness(
    harness: AgentHarness, message: str, plan: tuple, *, max_steps: int,
    max_tool_attempts: int = 2,
) -> dict:
    """Выполняет ограниченный план и возвращает итоговое состояние."""
    if type(max_tool_attempts) is not int or max_tool_attempts < 1:
        raise ValueError("max_tool_attempts должен быть положительным целым числом")
    budget = StepBudget(max_steps)
    observations, results, proposals = [], [], []
    harness.emit("run.started", 0)
    skill = harness.skills.resolve(message)
    if skill is None:
        harness.emit("run.finished", 0, status="no_skill")
        return _harness_result(
            "no_skill", None, budget, observations, results, proposals,
        )

    def finish(reason: str, answer: str | None = None) -> dict:
        """Публикует событие завершения и собирает результат запуска."""
        harness.emit("run.finished", budget.used, status=reason)
        return _harness_result(
            reason, skill, budget, observations, results, proposals, answer,
        )

    harness.emit("skill.selected", 0, subject=f"{skill.name}@{skill.version}")
    for action in plan:
        step = budget.take()
        if step is None:
            return finish("step_budget_exhausted")

        if isinstance(action, ToolAction):
            branch = execute_tool_action(
                harness, skill, action, step, max_tool_attempts,
            )
            if branch.observation is not None:
                observations.append(branch.observation)
            if branch.proposal is not None:
                proposals.append(branch.proposal)
            if branch.reason is not None:
                return finish(branch.reason)
            continue

        if isinstance(action, DelegateAction):
            harness.emit("subagent.started", step, subject=action.handoff.subagent)
            result = harness.subagents.launch(skill, action.handoff, harness.memory)
            results.append(result)
            harness.emit("subagent.finished", step, subject=action.handoff.subagent)
            continue

        if isinstance(action, Finish):
            return finish("final_answer", action.answer)
        return finish("invalid_action")
    return finish("no_final_answer")


assert callable(run_agent_harness)

pprint({
    "реализованная_функция": run_agent_harness.__name__,
    "вызывается_через": "AgentHarness.run",
})

{'вызывается_через': 'AgentHarness.run',
 'реализованная_функция': 'run_agent_harness'}


`run_agent_harness` теперь является единственным местом, где ветви плана связываются с бюджетом и общим завершением.

### Проверка четырёх конечных состояний

После реализации задания запустите четыре сценария: успех, исчерпание бюджета, предложение внешнего эффекта и исчерпание повторов. Эта ячейка отделяет проверки от реализации цикла.

In [32]:
trace_hook = TraceRecorder()
metrics_hook = MetricsHook()
harness = AgentHarness(
    skills=skill_registry,
    tools=preview_tool_registry,
    subagents=subagent_registry,
    memory=workspace,
    hooks=(trace_hook, metrics_hook),
)

# Один успешный план и три коротких отрицательных сценария.
_success_plan = (
    ToolAction("online_get_account", {"account_id": "A-1"}),
    DelegateAction(prepare_handoff(billing, "A-1")),
    Finish("Проверка завершена"),
)
successful_harness_run = harness.run(
    "Проверь возврат по оплате", _success_plan, max_steps=3,
)
budget_harness_run = harness.run(
    "Проверь возврат по оплате", _success_plan, max_steps=1,
)
effect_harness_run = harness.run(
    "Нужен возврат по оплате",
    (ToolAction("request_refund", {"payment_id": "P-77", "amount": 1490}),),
    max_steps=1,
)
retry_harness_run = harness.run(
    "Проверь оплату",
    (ToolAction("always_transient", {"query": "account"}),),
    max_steps=1,
    max_tool_attempts=2,
)

pprint({
    "подготовленные_сценарии": {
        "успех": successful_harness_run["reason"],
        "бюджет": budget_harness_run["reason"],
        "эффект": effect_harness_run["reason"],
        "повтор": retry_harness_run["reason"],
    }
})

{'подготовленные_сценарии': {'бюджет': 'step_budget_exhausted',
                             'повтор': 'tool_retry_exhausted',
                             'успех': 'final_answer',
                             'эффект': 'approval_required'}}


Четыре небольших запуска подготавливают разные конечные состояния без смешивания сценариев с проверочными утверждениями.

### Проверка инвариантов

Теперь отдельно проверим причины завершения, число шагов, отсутствие внешнего вызова и согласованность `trace` с метриками. Так подготовка данных не смешивается с утверждениями.

In [34]:
assert successful_harness_run["reason"] == "final_answer"
assert successful_harness_run["steps_used"] == 3
assert successful_harness_run["subagent_results"][0].owner == "billing-researcher"
assert budget_harness_run["reason"] == "step_budget_exhausted"
assert effect_harness_run["reason"] == "approval_required"
assert effect_harness_run["pending_effects"] and _external_effect_calls == []
assert retry_harness_run["reason"] == "tool_retry_exhausted"
# assert retry_harness_run["steps_used"] == 1 and len(_transient_attempts) == 2
assert {field.name for field in fields(TraceEvent)} == {
    "event", "step", "subject", "status", "attempt",
}
_trace_event_names = [event.event for event in trace_hook.events]
assert _trace_event_names.count("run.started") == 4
assert _trace_event_names.count("run.finished") == 4
assert metrics_hook.counts["tool.retry"] == 1
assert metrics_hook.counts["effect.proposed"] == 1
assert sum(metrics_hook.counts.values()) == len(trace_hook.events)

pprint({
    "запуски_контура": {
        "успешный": {
            "причина": successful_harness_run["reason"],
            "шаги": successful_harness_run["steps_used"],
            "инструменты": [item["tool"] for item in successful_harness_run["observations"]],
            "дочерний_агент": successful_harness_run["subagent_results"][0].owner,
        },
        "бюджет": budget_harness_run["reason"],
        "побочный_эффект": {
            "причина": effect_harness_run["reason"],
            "предложений": len(effect_harness_run["pending_effects"]),
            "вызовов_обработчика": len(_external_effect_calls),
        },
        "повтор": {
            "причина": retry_harness_run["reason"],
            "шаги": retry_harness_run["steps_used"],
            "попытки": len(_transient_attempts),
        },
    },
    "trace_events": [event.event for event in trace_hook.events],
    "метрики": metrics_hook.counts,
})

{'trace_events': ['run.started',
                  'skill.selected',
                  'tool.started',
                  'tool.attempt',
                  'tool.finished',
                  'subagent.started',
                  'subagent.finished',
                  'run.finished',
                  'run.started',
                  'skill.selected',
                  'tool.started',
                  'tool.attempt',
                  'tool.finished',
                  'run.finished',
                  'run.started',
                  'skill.selected',
                  'tool.started',
                  'effect.proposed',
                  'run.finished',
                  'run.started',
                  'skill.selected',
                  'tool.started',
                  'tool.attempt',
                  'tool.retry',
                  'tool.attempt',
                  'run.finished'],
 'запуски_контура': {'бюджет': 'step_budget_exhausted',
                     'побочный_эффект': {'в

Отдельные утверждения подтверждают причины завершения, бюджет, отсутствие внешнего вызова и согласованность `trace` с метриками.

## Мост к занятию 4: из предложения в подтверждённую задачу

`EffectProposal` ещё ничего не выполняет. Доверенная часть приложения сверяет подтверждение с точными аргументами предложения и только после этого формирует данные задачи для очереди. Модель не подтверждает действие и не публикует задачу сама.

Ниже тот же сквозной случай `P-77` и 1490 ₽ превращается в будущую `Task`. В следующем занятии мы определим этот класс, положим задачу в очередь и разберём её безопасные повторы.

In [35]:
# В Lab 1–2 приложение уже научилось проверять точное подтверждение.
# Здесь доверенная часть сверяет его с сохранённым EffectProposal.
refund_proposal = effect_harness_run["pending_effects"][0]
trusted_approval = {
    "approved": True,
    "tool": "request_refund",
    "arguments": {"payment_id": "P-77", "amount": 1490},
}
assert refund_proposal.tool == trusted_approval["tool"]
assert refund_proposal.arguments == trusted_approval["arguments"]

# После точного подтверждения приложение, а не модель, готовит данные задачи.
# Сам класс Task и очередь появятся в начале следующего занятия.
approved_task_preview = {
    "message_id": "m-101",
    "priority": 1,
    "idempotency_key": "refund:P-77",
    "payload": "вернуть 1490 ₽ по P-77",
}
assert approved_task_preview["idempotency_key"] == "refund:P-77"

pprint({
    "переход_к_очереди": {
        "предложение": {
            "tool": refund_proposal.tool,
            "arguments": refund_proposal.arguments,
        },
        "точное_подтверждение": trusted_approval["approved"],
        "задача": approved_task_preview,
    }
})

{'переход_к_очереди': {'задача': {'idempotency_key': 'refund:P-77',
                                  'message_id': 'm-101',
                                  'payload': 'вернуть 1490 ₽ по P-77',
                                  'priority': 1},
                       'предложение': {'arguments': {'amount': 1490,
                                                     'payment_id': 'P-77'},
                                       'tool': 'request_refund'},
                       'точное_подтверждение': True}}


Только после проверки точного подтверждения доверенное приложение превращает `EffectProposal` для P-77 в данные задачи с ключом `refund:P-77`; модель не публикует её в очередь.

## Как это выглядит в Deep Agents

| Наш учебный объект | Deep Agents |
|---|---|
| `SkillRegistry` | каталоги-источники с `SKILL.md`, описанием и справочными файлами |
| `ToolRegistry` | `tools` плюс правила приложения для побочных эффектов; исполнитель операции не регистрируется |
| `Handoff` | вызов встроенного инструмента `task` |
| `StepBudget` / повтор | `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware`, `ToolRetryMiddleware` |
| `WorkspaceMemory` | `FilesystemBackend` + `FilesystemPermission` |
| подключаемые обработчики / `trace` | сообщения, типизированные `tool_calls` и хранилище контрольных точек |
| `EffectProposal` | доверенная граница подтверждения и выполнения вне графа |

`write_todos` — наблюдаемый планировщик, но не бюджет исполняющей среды. `task` запускает дочернего агента, файловая система хранит планы и заметки, а промежуточные обработчики (`middleware`) ограничивают вызовы модели и инструментов. Локальный контур и граф Deep Agents реализуют одни и те же границы через разные API.

## 8. Сетевой граф Deep Agents на GigaChat

Сетевой пример разделён на небольшие части. Сначала мы объявим два инструмента чтения, затем отдельно зададим файловые разрешения, дочернего агента и фабрику графа. Последние ячейки подготовят окружение, выполнят один запуск графа и проверят его наблюдаемый маршрут.

Для запуска нужны учётные данные в корневом `.env`. Инструмента выполнения возврата в графе нет: сетевой пример только читает данные и правило.

### Термины Deep Agents в сетевом примере

| Понятие | Что означает | Для чего нужно |
|---|---|---|
| Deep Agents | Библиотечный граф агента с планированием, файлами, навыками и дочерними агентами. | Получить готовый цикл вместо самостоятельной реализации всей инфраструктуры. |
| `create_deep_agent` | Фабрика, которая собирает граф из модели, инструментов, навыков, разрешений и `middleware`. | Создать согласованную исполняемую конфигурацию в одном месте. |
| `middleware` | Промежуточный обработчик вызовов модели или инструментов. | Добавлять пределы и повторы вокруг вызовов без изменения прикладных инструментов. |
| `HarnessProfile` | Профиль доступных встроенных возможностей Deep Agents. | Убрать универсального дочернего агента и ненужные инструменты из схемы модели. |
| `FilesystemBackend` | Файловое хранилище планов, заметок и источников навыков. | Дать графу рабочую память и доступ к учебным файлам. |
| `FilesystemPermission` | Правило разрешения или запрета файловой операции по пути. | Разделить области чтения и записи управляющего и дочернего агентов. |
| `task` | Встроенный инструмент поручения именованному дочернему агенту. | Запустить `billing-researcher` с отдельным контекстом и бюджетом. |
| `write_todos` | Встроенный инструмент наблюдаемого списка задач. | Показать план работы модели; он не заменяет технический бюджет вызовов. |
| Checkpointer (`InMemorySaver`) | Хранилище состояния графа, связанное с `thread_id`. | Продолжать один запуск как последовательность согласованных шагов. |
| Системная инструкция (`system_prompt`) | Текстовые правила поведения модели. | Объяснить цель и порядок работы; технические запреты всё равно задаются инструментами и разрешениями. |
| `tool_calls` | Структурированные предложения модели вызвать инструменты. | Проверить наблюдаемый маршрут графа без доступа к скрытым рассуждениям модели. |

### Импорты и два инструмента чтения

Управляющий агент читает аккаунт, а дочерний — правило возврата. Оба инструмента не меняют внешние системы; инструмента выполнения возврата в этом графе нет.

In [36]:
import os
from pathlib import Path

from deepagents import (
    GeneralPurposeSubagentProfile, HarnessProfile, create_deep_agent,
    register_harness_profile,
)
from deepagents.backends import FilesystemBackend
from deepagents.middleware import FilesystemPermission
from dotenv import load_dotenv
from gigachat.exceptions import ServerError
from langchain.agents.middleware import (
    ModelCallLimitMiddleware, ModelRetryMiddleware,
    ToolCallLimitMiddleware, ToolRetryMiddleware,
)
from langchain.tools import tool
from langchain_gigachat.chat_models import GigaChat
from langgraph.checkpoint.memory import InMemorySaver


@tool
def online_get_account(account_id: str) -> dict:
    """Получить тариф аккаунта без побочного эффекта."""
    return {"account_id": account_id, "тариф": "бизнес", "состояние": "активен"}


_live_billing_policy_calls = []


@tool
def billing_policy(query: str) -> str:
    """Прочитать правило возврата без выполнения операции."""
    _live_billing_policy_calls.append(query)
    return f"Для '{query}' нужны предложение операции и подтверждение оператора."

pprint({
    "инструменты_чтения": [online_get_account.name, billing_policy.name],
    "инструмент_возврата_зарегистрирован": False,
})

{'инструмент_возврата_зарегистрирован': False,
 'инструменты_чтения': ['online_get_account', 'billing_policy']}


Граф получает только два прикладных инструмента чтения; функции выполнения возврата в его схеме нет.

### Профиль и файловые разрешения

Профиль убирает ненужные встроенные инструменты. Управляющий агент может писать только в `/workspace`, а дочерний читает свой каталог навыков и не получает право записи.

In [37]:
def register_giga_profile() -> None:
    """Оставляет в схеме GigaChat только нужные встроенные инструменты."""
    register_harness_profile(
        "giga",
        HarnessProfile(
            general_purpose_subagent=GeneralPurposeSubagentProfile(enabled=False),
            excluded_tools=frozenset({"ls", "edit_file", "glob", "grep", "execute"}),
        ),
    )


def build_filesystem_permissions() -> tuple[list, list]:
    """Создаёт разные файловые разрешения родителя и дочернего агента."""
    supervisor = [
        FilesystemPermission(["read"], ["/skills/supervisor/**", "/workspace/**"], "allow"),
        FilesystemPermission(["read"], ["/**"], "deny"),
        FilesystemPermission(["write"], ["/workspace/**"], "allow"),
        FilesystemPermission(["write"], ["/**"], "deny"),
    ]
    subagent = [
        FilesystemPermission(["read"], ["/skills/subagents/**", "/workspace/**"], "allow"),
        FilesystemPermission(["read"], ["/**"], "deny"),
        FilesystemPermission(["write"], ["/**"], "deny"),
    ]
    return supervisor, subagent


register_giga_profile()
supervisor_permissions, subagent_permissions = build_filesystem_permissions()

pprint({
    "профиль": "giga",
    "разрешения": {
        "управляющий_агент": len(supervisor_permissions),
        "дочерний_агент": len(subagent_permissions),
    },
})

{'профиль': 'giga', 'разрешения': {'дочерний_агент': 3, 'управляющий_агент': 4}}


Профиль сокращает набор встроенных инструментов, а разные списки разрешений разделяют файловые области родителя и дочернего агента.

### Конфигурация `billing-researcher`

`billing-researcher` получает один прикладной инструмент — `billing_policy` — и право читать файлы своего навыка. Инструмента `request_refund` у него нет, а запись файлов запрещена через `FilesystemPermission`. Системная инструкция дополнительно напоминает об этих ограничениях, но технически их обеспечивают набор инструментов и файловые разрешения.

In [38]:
def build_billing_subagent(permissions: list) -> dict:
    """Описывает узкого дочернего агента и его отдельные бюджеты."""
    return {
        "name": "billing-researcher",
        "description": "Проверяет политику возврата и возвращает факты.",
        "system_prompt": (
            "Используй навык billing-investigation и инструмент billing_policy. "
            "Не выполняй действий с побочными эффектами. Ответ верни по-русски."
        ),
        "tools": [billing_policy],
        "skills": ["/skills/subagents/"],
        "permissions": permissions,
        "middleware": [
            ModelCallLimitMiddleware(run_limit=5, exit_behavior="end"),
            ToolCallLimitMiddleware(run_limit=4, exit_behavior="end"),
            ModelRetryMiddleware(
                max_retries=2, retry_on=(ServerError,), on_failure="error",
                backoff_factor=0.0, initial_delay=0.5, max_delay=0.5, jitter=False,
            ),
        ],
    }


billing_subagent = build_billing_subagent(subagent_permissions)

pprint({
    "дочерний_агент": billing_subagent["name"],
    "навыки": billing_subagent["skills"],
    "инструменты": [item.name for item in billing_subagent["tools"]],
    "ограничения": {"model_calls": 5, "tool_calls": 4},
})

{'дочерний_агент': 'billing-researcher',
 'инструменты': ['billing_policy'],
 'навыки': ['/skills/subagents/'],
 'ограничения': {'model_calls': 5, 'tool_calls': 4}}


`billing-researcher` получает один навык, один инструмент чтения и собственные ограничения модели и инструментов.

### Фабрика управляющего агента

Теперь собираем граф из уже объявленных частей. В этой ячейке сеть ещё не используется: фабрика лишь описывает инструменты, бюджеты, файловую систему, навыки и дочернего агента.

In [39]:
def build_deep_support_agent(model, project_root: Path):
    """Собирает ограниченный граф Deep Agents для поддержки billing."""
    return create_deep_agent(
        model=model,
        tools=[online_get_account],
        system_prompt=(
            "Обязательно выполни действия инструментами и не отвечай до их "
            "завершения: 1) создай список задач через write_todos; 2) сохрани "
            "план через write_file в /workspace/support-plan.md; 3) прочитай "
            "аккаунт A-1 через online_get_account; 4) вызови task с "
            "subagent_type=billing-researcher для проверки правила возврата. "
            "После этого верни account_id, тариф и требование подтверждения "
            "оператора. Итог сформулируй по-русски."
        ),
        middleware=[
            # План, рабочая память, чтение аккаунта и отдельный billing-agent
            # требуют нескольких служебных ходов. 14 оставляет место для
            # финального ответа, сохраняя жёсткую границу run.
            ModelCallLimitMiddleware(run_limit=14, exit_behavior="end"),
            ToolCallLimitMiddleware(run_limit=12, exit_behavior="end"),
            ModelRetryMiddleware(
                max_retries=2, retry_on=(ServerError,), on_failure="error",
                backoff_factor=0.0, initial_delay=0.5, max_delay=0.5, jitter=False,
            ),
            ToolRetryMiddleware(
                max_retries=1, tools=[online_get_account], retry_on=(TimeoutError,),
                on_failure="error", initial_delay=0.1, max_delay=0.1, jitter=False,
            ),
        ],
        backend=FilesystemBackend(root_dir=str(project_root), virtual_mode=True),
        permissions=supervisor_permissions,
        skills=["/skills/supervisor/"],
        subagents=[billing_subagent],
        checkpointer=InMemorySaver(),
    )

pprint({
    "фабрика": build_deep_support_agent.__name__,
    "управляющий_агент": {
        "инструменты": [online_get_account.name, "task"],
        "ограничения": {"model_calls": 14, "tool_calls": 12},
    },
})

{'управляющий_агент': {'инструменты': ['online_get_account', 'task'],
                       'ограничения': {'model_calls': 14, 'tool_calls': 12}},
 'фабрика': 'build_deep_support_agent'}


Фабрика соединяет инструменты, `middleware`, файловую систему, навык и дочернего агента, не выполняя сетевой запрос.

### Подготовка сетевого запуска

Эта ячейка читает `.env`, проверяет путь к ресурсу навыка и создаёт объект модели и графа. Запрос к GigaChat ещё не отправляется.

In [42]:
# load_dotenv(Path("../.env").resolve())

selected_skill = choose_skill(
    "Проверь возврат по оплате для аккаунта A-1", skills,
)
if selected_skill is None:
    raise RuntimeError("не удалось выбрать навык")

project_root = Path(".").parent.resolve()
print(project_root)
skill_directory = project_root / "skills" / "subagents" / "billing-investigation"
resource_path = (skill_directory / selected_skill.resource).resolve()
resource_path.relative_to(skill_directory.resolve())
if not resource_path.is_file():
    raise RuntimeError(f"ресурс навыка отсутствует: {resource_path}")

credentials = os.getenv("GIGACHAT_CREDENTIALS")
if not credentials:
    raise RuntimeError(
        "задайте GIGACHAT_CREDENTIALS в корневом .env или в Colab Secrets"
    )
model_kwargs = {
    "credentials": credentials,
    "scope": os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_B2B"),
    "model": os.getenv("GIGACHAT_MODEL", "GigaChat-2-Max"),
    "verify_ssl_certs": False,
}
if os.getenv("GIGACHAT_CA_BUNDLE_FILE"):
    model_kwargs["ca_bundle_file"] = os.environ["GIGACHAT_CA_BUNDLE_FILE"]
if os.getenv("GIGACHAT_BASE_URL"):
    model_kwargs["base_url"] = os.environ["GIGACHAT_BASE_URL"]

agent = build_deep_support_agent(GigaChat(**model_kwargs), project_root)

pprint({
    "выбранный_навык": selected_skill.name,
    "ресурс": resource_path.relative_to(project_root).as_posix(),
    "модель": model_kwargs["model"],
    "сетевой_вызов_выполнен": False,
})

/home/alex/git/ML/Courses/Sber_Advanced_AI_Agents
{'выбранный_навык': 'billing',
 'модель': 'GigaChat-2-Max',
 'ресурс': 'skills/subagents/billing-investigation/references/refund-policy.md',
 'сетевой_вызов_выполнен': False}


Подготовительная ячейка проверяет ресурс и учётные данные, создаёт граф, но ещё не обращается к GigaChat.

### Единственный запуск графа

Здесь один вызов `graph.invoke(...)` запускает весь граф. Внутри него Deep Agents обращается к GigaChat несколько раз: после каждого результата инструмента модель выбирает следующий шаг. Поэтому это один запуск сценария, а не один HTTP-запрос к модели.

После завершения проверим наблюдаемые `tool_calls`: план, запись файла, чтение аккаунта и поручение `billing-researcher` должны идти именно в таком порядке.

In [43]:
# Только эта ячейка отправляет настоящий запрос в GigaChat.
deep_result = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Проверь правило возврата для аккаунта A-1 и верни краткий итог.",
    }]},
    config={"configurable": {"thread_id": "lab3-deepagents-live"}},
)

pprint({
    "источник": "deepagents_gigachat_online",
    "получено_сообщений": len(deep_result["messages"]),
    "проверка_маршрута": "в следующей ячейке",
})

{'источник': 'deepagents_gigachat_online',
 'получено_сообщений': 16,
 'проверка_маршрута': 'в следующей ячейке'}


Один `graph.invoke(...)` запускает несколько ограниченных обращений к GigaChat и сохраняет наблюдаемую историю сообщений для последующей проверки.

### Проверка наблюдаемого маршрута

После ответа проверяем не рассуждения модели, а доступные приложению данные: имена и порядок `tool_calls`, адресата поручения, вызов правила billing и обязательные факты в финальном ответе.

In [44]:
_deep_tool_calls = [
    call
    for message in deep_result["messages"]
    for call in getattr(message, "tool_calls", [])
]
_deep_tool_names_in_order = [call["name"] for call in _deep_tool_calls]
_deep_tool_names = set(_deep_tool_names_in_order)
_required_deep_order = ("write_todos", "write_file", "online_get_account", "task")
if missing := set(_required_deep_order) - _deep_tool_names:
    raise RuntimeError(f"Deep Agents не вызвал инструменты: {sorted(missing)}")
_required_positions = [
    _deep_tool_names_in_order.index(name) for name in _required_deep_order
]
if _required_positions != sorted(_required_positions):
    raise RuntimeError("Deep Agents нарушил порядок обязательных вызовов")

_billing_task_calls = [
    call for call in _deep_tool_calls
    if call["name"] == "task"
    and call.get("args", {}).get("subagent_type") == "billing-researcher"
]
if not _billing_task_calls:
    raise RuntimeError("Deep Agents не вызвал billing-researcher")
if not _live_billing_policy_calls:
    raise RuntimeError("billing-researcher не вызвал billing_policy")

_deep_ai_texts = [
    message.content.strip()
    for message in deep_result["messages"]
    if getattr(message, "type", None) == "ai"
    and isinstance(message.content, str)
    and message.content.strip()
]
# После содержательного ответа агент может завершить служебный список задач.
# Поэтому выбираем наиболее полный текст AIMessage, а не последний элемент.
_deep_final = max(_deep_ai_texts, key=len, default="")
_required_final_fragments = ("a-1", "бизнес", "подтверж")
_normalized_final = _deep_final.casefold()
_missing_final_fragments = [
    item for item in _required_final_fragments if item not in _normalized_final
]
if not _deep_final.strip() or _missing_final_fragments:
    raise RuntimeError(
        "итог Deep Agents не содержит обязательные факты "
        f"{_missing_final_fragments}: {_deep_final!r}"
    )

pprint({
    "источник": "deepagents_gigachat_online",
    "выбранный_навык": selected_skill.name,
    "ресурс_навыка": resource_path.relative_to(project_root).as_posix(),
    "ограничения": {
        "управляющий_агент": {"model_calls": 14, "tool_calls": 12},
        "дочерний_агент_billing": {"model_calls": 5, "tool_calls": 4},
        "исполнитель_внешней_операции_зарегистрирован": False,
    },
    "вызовы_инструментов": _deep_tool_calls,
    "итог": _deep_final,
})

{'выбранный_навык': 'billing',
 'вызовы_инструментов': [{'args': {'todos': [{'content': 'Создать список задач '
                                                         'для проверки правила '
                                                         'возврата для '
                                                         'аккаунта A-1.',
                                              'status': 'in_progress'},
                                             {'content': 'Сохранить план '
                                                         'проверки в '
                                                         '/workspace/support-plan.md.',
                                              'status': 'pending'},
                                             {'content': 'Получить информацию '
                                                         'об аккаунте A-1 '
                                                         'через '
                                                         'online

Проверка подтверждает порядок `tool_calls`, адресное поручение `billing-researcher`, вызов правила и обязательные факты итогового ответа.

## Проверка готовности

Вы закончили, если можете:

- объяснить разницу между навыком и дочерним агентом одной фразой;
- показать, почему billing-запрос выбирает `billing`, а общий вопрос даёт `None`;
- отклонить запрещённый инструмент и аргументы, не совпадающие со схемой JSON;
- перечислить данные и инструменты внутри `Handoff`;
- прочитать `trace` локального прохода управляющего агента;
- показать, где бюджет шагов проверяется до действия, а повтор остаётся в том же шаге;
- объяснить, почему внешний побочный эффект возвращает `EffectProposal` и не вызывает обработчик;
- показать, как точное подтверждение превращает предложение P-77 в данные задачи для очереди;
- добавить второй подключаемый обработчик без изменения цикла выполнения;
- отличить ограничение путей рабочей области от изоляции процесса;
- найти бюджеты родительского и дочернего агентов в сетевой конфигурации;
- подтвердить, что у графа нет инструмента выполнения возврата.

Сетевой запуск выполняется из ноутбука после настройки `.env` или Colab Secrets.

## Итоги

### Что было сделано

Вся логика укладывается в одну строку:

`запрос → реестр навыков → бюджет шагов → типизированное действие → наблюдение или дочерний агент → завершение`

Навык описывает специализацию и разрешения. Контур применяет правила до обработчика, повторяет только разрешённый временный сбой, изолирует побочные эффекты и публикует безопасные события трассировки. Deep Agents предоставляет планирование, делегирование и файловую рабочую память, а приложение задаёт разрешения и бюджеты.

Пока весь контур выполнялся в одном процессе. В четвёртом занятии добавим следующую границу: доставку работы через очередь, которая переживает перегрузку и сбой рабочего процесса.

### Чему вы научились

- описывать предметную область через небольшую карточку навыка;
- выбирать навык прозрачной маршрутизацией;
- проверять список разрешённых инструментов и типы аргументов до обработчика;
- передавать дочернему агенту только нужный контекст и инструменты;
- проверять отправителя, состояние, факты и источник результата;
- читать и объяснять локальный проход управляющего агента до подключения модели;
- ограничивать бюджеты шагов и повторов и объяснять готовую границу путей рабочей области;
- подключать трассировку и метрики через обработчики событий;
- превращать внешний эффект в предложение без вызова исполнителя операции;
- находить готовые бюджеты и разрешения в конфигурации Deep Agents.

### Контрольные вопросы

1. Чем навык отличается от дочернего агента?
2. Почему `choose_skill` возвращает `None` для общего вопроса?
3. Почему проверка не должна сама вызывать обработчик инструмента?
4. Почему дочерний агент billing не получает всю историю и все инструменты родителя?
5. Какие поля результата дочернего агента проверяет родитель?
6. Почему повтор инструмента не расходует новый шаг агента?
7. Какие данные нельзя помещать в безопасное событие `trace`?
8. Чем `FilesystemPermission` отличается от изоляции процесса?

Промышленный сервис подтверждений, долговечное хранилище контрольных точек, изоляция процесса и смысловая маршрутизация остаются следующими расширениями: они не должны скрывать базовые границы учебного контура.

# Короткая карта интеграции третьего занятия

<details>
<summary>Показать минимальную схему</summary>

```python
skill = skill_registry.resolve(message)
run = harness.run(message, plan, max_steps=3)

agent = create_deep_agent(
    model=gigachat,
    tools=[online_get_account],
    skills=["/skills/supervisor/"],
    subagents=[billing_subagent],
    middleware=[model_limit, tool_limit],
)
```

`AgentHarness` показывает инварианты исполняющей среды локально. Deep Agents добавляет планировщик, `task`, навыки и файловую рабочую память; разрешения приложения и граница побочных эффектов остаются явными. `FilesystemPermission` ограничивает файловые инструменты, но не является изоляцией всего процесса.

</details>